In [ ]:
import torch
import torch.nn as nn
from harl.utils.models_tools import get_init_method

In [ ]:
"""RNN modules."""
class RNNLayer(nn.Module):
    def __init__(self, inputs_dim, outputs_dim, recurrent_n, initialization_method):
        super(RNNLayer, self).__init__()
        self.recurrent_n = recurrent_n
        self.initialization_method = initialization_method

        self.rnn = nn.GRU(inputs_dim, outputs_dim, num_layers=self.recurrent_n)
        for name, param in self.rnn.named_parameters():
            if "bias" in name:
                nn.init.constant_(param, 0)
            elif "weight" in name:
                init_method = get_init_method(initialization_method)
                init_method(param)
        self.norm = nn.LayerNorm(outputs_dim)

    def forward(self, x, hxs, masks):
        if x.size(0) == hxs.size(0):
            x, hxs = self.rnn(
                x.unsqueeze(0),
                (hxs * masks.repeat(1, self.recurrent_n).unsqueeze(-1))
                .transpose(0, 1)
                .contiguous(),
            )
            x = x.squeeze(0)
            hxs = hxs.transpose(0, 1)
        else:
            # x is a (T, N, -1) tensor that has been flatten to (T * N, -1)
            N = hxs.size(0)
            T = int(x.size(0) / N)

            # unflatten
            x = x.view(T, N, x.size(1))

            # Same deal with masks
            masks = masks.view(T, N)

            # Let's figure out which steps in the sequence have a zero for any agent
            # We will always assume t=0 has a zero in it as that makes the logic cleaner
            has_zeros = (masks[1:] == 0.0).any(dim=-1).nonzero().squeeze().cpu()

            # +1 to correct the masks[1:]
            if has_zeros.dim() == 0:
                # Deal with scalar
                has_zeros = [has_zeros.item() + 1]
            else:
                has_zeros = (has_zeros + 1).numpy().tolist()

            # add t=0 and t=T to the list
            has_zeros = [0] + has_zeros + [T]

            hxs = hxs.transpose(0, 1)

            outputs = []
            for i in range(len(has_zeros) - 1):
                # We can now process steps that don't have any zeros in masks together!
                # This is much faster
                start_idx = has_zeros[i]
                end_idx = has_zeros[i + 1]
                temp = (
                    hxs * masks[start_idx].view(1, -1, 1).repeat(self.recurrent_n, 1, 1)
                ).contiguous()
                rnn_scores, hxs = self.rnn(x[start_idx:end_idx], temp)
                outputs.append(rnn_scores)

            # assert len(outputs) == T
            # x is a (T, N, -1) tensor
            x = torch.cat(outputs, dim=0)

            # flatten
            x = x.reshape(T * N, -1)
            hxs = hxs.transpose(0, 1)

        x = self.norm(x)
        return x, hxs


In [ ]:
import torch
from torch import nn as nn
from torch import optim as optim
import torch.utils.data as data
import math
import copy

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class CrossAgentAttention(nn.Module):
    """Manual inter-agent cross-attention mechanism."""
    def __init__(self, d_model=128, n_heads=4, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.dk = d_model // n_heads
        self.scale = self.dk ** -0.5

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None, attention_bias=None):
        """
        x: (B, N, d_model)
        mask: (B, N) binary mask (1=valid, 0=padding)
        attention_bias: (B, N, N) optional attention bias
        """
        B, N, _ = x.shape

        q = self.q_proj(x).view(B, N, self.n_heads, self.dk).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_heads, self.dk).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_heads, self.dk).transpose(1, 2)

        attn_logits = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        if attention_bias is not None:
            attn_logits = attn_logits + attention_bias.unsqueeze(1)

        if mask is not None:
            mask_ = mask.unsqueeze(1).unsqueeze(2)  # (B, 1, 1, N)
            attn_logits = attn_logits.masked_fill(mask_ == 0, -1e9)

        attn = F.softmax(attn_logits, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, N, self.d_model)
        out = self.out_proj(out)
        out = self.norm(x + out)

        return out, attn

In [2]:
class ObsOnlyCrossAttentionCritic(nn.Module):
    def __init__(self, obs_dim, num_agent_types, d_model=128, n_heads=4):
        super().__init__()
        self.obs_proj = nn.Linear(obs_dim, d_model)
        self.type_embed = nn.Embedding(num_agent_types, d_model)

        # Cross-agent attention
        self.cross_attn = CrossAgentAttention(d_model=d_model, n_heads=n_heads)

        # Feedforward network
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model),
            nn.LayerNorm(d_model),
        )

        # Critic head (aggregated value)
        self.value_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, obs, agent_types, mask=None, attention_bias=None):
        """
        obs: (B, N, obs_dim)
        agent_types: (B, N)
        mask: (B, N)
        attention_bias: (B, N, N)
        """
        x = self.obs_proj(obs)
        x = x + self.type_embed(agent_types)

        # Inter-agent attention
        x, attn = self.cross_attn(x, mask, attention_bias)
        x = self.ff(x)

        # Aggregate across agents
        if mask is not None:
            pooled = (x * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp(min=1e-6)
        else:
            pooled = x.mean(dim=1)

        value = self.value_head(pooled)
        return value, attn

In [8]:
critic = ObsOnlyCrossAttentionCritic(obs_dim=18, num_agent_types=2)

# Case 1: 3 agents
obs3 = torch.randn(1, 3, 18)
types3 = torch.tensor([[1, 1, 0]])
mask3 = torch.ones(1, 3)

# Case 2: 5 agents
obs5 = torch.randn(1, 5, 18)
types5 = torch.tensor([[0, 1, 1, 0, 0]])
mask5 = torch.ones(1, 5)

value3, attn3 = critic(obs3, types3, mask3)
value5, attn5 = critic(obs5, types5, mask5)

print(value3.shape, value5.shape)

torch.Size([1, 1]) torch.Size([1, 1])


In [15]:
torch.linspace(0,1,2)

tensor([0., 1.])

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        output = self.W_o(self.combine_heads(attn_output))
        return output

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x